# Imports

#### Loading the data from Kaggle 

In [63]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
import polars as pl
from sklearn.preprocessing import OneHotEncoder

#model saving
import joblib
import pickle

from sklearn.base import BaseEstimator, TransformerMixin

## Loading the data

In [64]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kundanbedmutha/exam-score-prediction-dataset")

print("Path to dataset files:", path)

Path to dataset files: /home/ender/.cache/kagglehub/datasets/kundanbedmutha/exam-score-prediction-dataset/versions/2


In [65]:
df_raw = pd.read_csv(f"{path}/Exam_Score_Prediction.csv")

In [66]:
df_raw['status'] = df_raw['exam_score'].apply(lambda x : "pass" if x >= 60 else "fail")
df_raw['status'] = np.where(df_raw['exam_score'] >= 60, "pass", "fail")

In [67]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        20000 non-null  int64  
 1   age               20000 non-null  int64  
 2   gender            20000 non-null  object 
 3   course            20000 non-null  object 
 4   study_hours       20000 non-null  float64
 5   class_attendance  20000 non-null  float64
 6   internet_access   20000 non-null  object 
 7   sleep_hours       20000 non-null  float64
 8   sleep_quality     20000 non-null  object 
 9   study_method      20000 non-null  object 
 10  facility_rating   20000 non-null  object 
 11  exam_difficulty   20000 non-null  object 
 12  exam_score        20000 non-null  float64
 13  status            20000 non-null  object 
dtypes: float64(4), int64(2), object(8)
memory usage: 2.1+ MB


In [68]:
df_raw.head(5)

,student_id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score,status
0,1,17,male,diploma,2.78,92.9,yes,7.4,poor,coaching,low,hard,58.9,fail
1,2,23,other,bca,3.37,64.8,yes,4.6,average,online videos,medium,moderate,54.8,fail
2,3,22,male,b.sc,7.88,76.8,yes,8.5,poor,coaching,high,moderate,90.3,pass
3,4,20,other,diploma,0.67,48.4,yes,5.8,average,online videos,low,moderate,29.7,fail
4,5,20,female,diploma,0.89,71.6,yes,9.8,poor,coaching,low,moderate,43.7,fail


# Pipelines

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `study_hours`, `class_attendance`, `sleep_hours`, `sleep_quality`, `exam_difficulty`, `gender`, `status` from the dataset.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [104]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass 
    def fit(self, X, y=None):
        return self

        
    def transform(self, X):
        X = X.copy()
        X = X[['study_hours', 'class_attendance', 'sleep_hours', 'sleep_quality', 'exam_difficulty', 'gender', 'status']].copy()
        return X

In [159]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_col):
        self.target_col = target_col
        self.cat_features = None
        self.encoder = None

    def fit(self, X, y=None):
        X = X.copy()
        self.cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
        if self.target_col in self.cat_features:
            self.cat_features.remove(self.target_col)
        self.encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        self.encoder.fit(X[self.cat_features])
        return self

    
    def transform(self, X):
        X = X.copy()
        y = X[self.target_col] if self.target_col in X.columns else None
        encoded = self.encoder.transform(X[self.cat_features])
        encoded_df = pd.DataFrame(
            encoded,
            columns = self.encoder.get_feature_names_out(self.cat_features),
            index = X.index,
        )
        X = X.drop(columns=self.cat_features)
        X = pd.concat([X, encoded_df], axis=1)
        
        return (X.drop(columns=[self.target_col], errors='ignore'), y)
        

In [160]:
class TrainValidationTest:
    def init(self, test_size=0.2, random_state=21, stratify=True, valid_size=0.25):
        self.test_size = test_size
        self.random_state = random_state
        self.stratify = stratify
        self.valid_size = valid_size

    def split(self, X, y):
        starify_y = y if self.stratify else None
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, 
            test_size = self.test_size, 
            random_state = self.random_state, 
            starify = stratify_y
        )
        valid_ratio = self.valid_size / (1 - self.test_size)
        X_train, X_valid, y_train, y_valid = train_test_slit(
            X_temp, y_temp,
            test_size = valid_ratio,
            random_state = self.random_state,
            stratify = y_temp if self.stratify else None
        )

        return X_train, X_valid, X_test, y_train, y_valid, y_test
        

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `X_valid`, `y_train`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
``` 

In [178]:
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.results = []

    def choose(self, X_train, X_valid, y_train, y_valid):
        best_valid_acc = 0
        best_model = None
        best_model_name = None

        for idx, grid in grids:
            name = grid_dict[idx]
            print(f"\nEstimator: {name}")

            grid.fit(X_train, y_train)
            print(f"Best params: {grid.best_params_}")

            valid_pred = grid.predict(X_valid)
            valid_acc = accuracy_score(y_valid, valid_pred)
            print(f"Valid accuracy (accuracy_score): {valid_acc:.3f}")
            print(f"Best training accuracy (cross-validated): {grid.best_score_:.3f}")
            print(f"Validation set accuracy score for best params: {grid.score(X_valid, y_valid):.3f}")

            if valid_acc > best_valid_acc:
                best_valid_acc = valid_acc
                best_model_name = name
                best_model = grid.best_estimator_

            results.append({
                'name':name,
                'accuracy':valid_acc,
                'params':grid.best_params_,
            })

        print(f"\nClassifier with the best accuracy score: {best_model_name}")
        return best_model_name, best_model

    def best_results(self):
        return pd.DataFrame(self.results)

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [180]:
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator

    def final_score(self, X_train, y_train, X_test, y_test):
        estimator.fit(X_train, y_train)
        pred = estimator.predict(X_test)
        acc = accuracy_score(y_test, pred)
        print(f"Accuracy of the final model is {acc}")
        return acc

    def save_model(self, path):
        joblib.dump(self.estimator, path)
        print(f"Model was successfully saved to {path}")
        

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!